# RAG Testing on Internal Product Document (PDF)

#### Project: ShopSphere Assistant - Conversational AI Chatbot
#### Scenario: Testing the RAG pipeline against a real product requirements document (PRD) stored as a PDF, instead of fetching content from the internet.

**Document under test:** `ShopSphere_Assistant_PRD.pdf` (internal PRD, version 2.3)

**Pipeline:** Load PDF -> Chunk -> Embed (nomic-embed-text) -> Store in Chroma -> Retrieve top-k -> Generate answer with llama3.2 -> Evaluate with DeepEval

![RAG Testing](img/RagTesting.png)

In [1]:
!pip install -qU langchain langchain-ollama langchain-chroma langchain-community langchain-text-splitters pypdf deepeval

#### Imports required for the PDF-based RAG pipeline

In [2]:
from pathlib import Path

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from typing import List

C:\Users\md twasif hossain\AppData\Local\Temp\ipykernel_14564\1636097448.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


#### Project configuration

All paths and model settings live in one place so the pipeline can be pointed at any internal document without touching the rest of the code.

In [3]:
# ---- Configuration ----
DOCUMENT_PATH = Path("ShopSphere_Assistant_PRD.pdf")   # internal PRD under test
CHUNK_SIZE = 500
CHUNK_OVERLAP = 0
TOP_K = 2

EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434"
COLLECTION_NAME = "shopsphere_prd_v23"   # one Chroma collection per document version

#### Setup the Ollama LLM (runs fully on-premise, no data leaves the company)

In [4]:
llm = ChatOllama(
    base_url=OLLAMA_BASE_URL,
    model=LLM_MODEL,
    temperature=0.5,
    max_tokens=512,
)

#### Step 1: Load the PDF document

`PyPDFLoader` reads the local PRD file. Each page becomes one `Document`.

In [5]:
loader = PyPDFLoader(str(DOCUMENT_PATH))
data = loader.load()

print(f"Loaded {len(data)} pages from {DOCUMENT_PATH.name}")
print("Page 1 preview:", data[0].page_content[:250], "...")

Loaded 4 pages from ShopSphere_Assistant_PRD.pdf
Page 1 preview: ShopSphere Assistant
Conversational AI Shopping Assistant - Product Requirements Document
Document ID
SS-PRD-2025-041
Version
2.3
Status
Approved for Pilot
Owner
Product Engineering - AI Platform Team
Release Target
Q3 2025 (Phased rollout)
Classific ...


#### Step 2: Chunk the document, embed and store in the vector DB

In [7]:
# Split the document into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)
chunks = text_splitter.split_documents(data)
print(f"Created {len(chunks)} chunks")

# Embed the chunks and store them in Chroma
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
vectorDB = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
)

# Create a retriever that returns the top-k most similar chunks
retriever = vectorDB.as_retriever(search_kwargs={"k": TOP_K})

Created 21 chunks


#### Step 3: Build the prompt and the generation chain

The prompt tells the model to answer **only from the retrieved context** - this is the grounding rule.

In [8]:
def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in docs])

template = """You are ShopSphere Assistant, a helpful customer-support chatbot.
Answer the question using ONLY the context provided below. If the context does not contain the answer, say you do not know.
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def retrieve_and_format(question: str) -> str:
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = prompt | llm | StrOutputParser()

#### Step 4: Ask the assistant questions about the internal document

In [9]:
def ask(question: str) -> str:
    context = retrieve_and_format(question)
    return chain.invoke({"context": context, "question": question})

In [10]:
# Smoke test - three questions covering different sections of the PRD
for q in [
    "What is ShopSphere Assistant?",
    "How long does a refund take?",
    "When is a conversation handed over to a human agent?",
]:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

Q: What is ShopSphere Assistant?
A: ShopSphere Assistant is a conversational AI chatbot embedded across the ShopSphere e-commerce platform (web, iOS, and Android), helping shoppers find products, track orders, manage returns, and more.

Q: How long does a refund take?
A: Credit card refunds are processed within 5 business days. Digital wallet refunds are processed within 24 hours after the returned item is received at the warehouse.

Q: When is a conversation handed over to a human agent?
A: Conversations are handed over to a human agent when the confidence score is below 0.55, or when there are two consecutive misunderstood messages, or when there are mentions of payment disputes or legal complaints, or when there is an explicit request.



### Flow: Testing the RAG application with DeepEval

![image.png](attachment:image.png)

#### Setup the golden dataset

Expected outputs are written by the QA team and taken directly from the PRD.

In [11]:
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

test_cases = [
    LLMTestCase(
        name="What is ShopSphere Assistant?",
        input="What is ShopSphere Assistant?",
        actual_output=ask("What is ShopSphere Assistant?"),
        expected_output="ShopSphere Assistant is an AI chatbot on the ShopSphere website and mobile apps that helps customers find products, track orders and handle returns using a retrieval-augmented generation (RAG) system built on the llama3.2 model with a local vector database.",
    ),
    LLMTestCase(
        name="Refund timeline",
        input="How long does a refund take?",
        actual_output=ask("How long does a refund take?"),
        expected_output="Credit card refunds are processed within 5 business days. Digital wallet refunds are processed within 24 hours after the returned item is received at the warehouse.",
    ),
    LLMTestCase(
        name="Human handoff trigger",
        input="When is a conversation handed over to a human agent?",
        actual_output=ask("When is a conversation handed over to a human agent?"),
        expected_output="The conversation is handed over to a live agent when the confidence score is below 0.55, after two consecutive misunderstood messages, when the customer mentions payment disputes or legal complaints, or when the customer explicitly asks for a human.",
    ),
]

dataset = EvaluationDataset()
for tc in test_cases:
    dataset.add_test_case(test_case=tc)

In [12]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is ShopSphere Assistant?', actual_output='ShopSphere Assistant is a conversational AI chatbot embedded across the ShopSphere e-commerce platform - web, iOS and Android. It helps shoppers find products, track orders, manage returns and other shopping-related tasks.', expected_output='ShopSphere Assistant is an AI chatbot on the ShopSphere website and mobile apps that helps customers find products, track orders and handle returns using a retrieval-augmented generation (RAG) system built on the llama3.2 model with a local vector database.', context=None, retrieval_context=None, metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, completion_time=None, flaky=False, multimodal=False, name='What is ShopSphere Assistant?', tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None, ex

#### Define evaluation metrics (DeepEval 4.x)

In [13]:
from deepeval.models import OllamaModel
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval

evaluator_model = OllamaModel(model="llama3.2:latest", base_url=OLLAMA_BASE_URL)

concise_metrics = GEval(
    name="Conciseness and Correctness",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    criteria="Evaluate whether the actual output is concise and correct compared to the expected output. The actual output should be a brief summary of the expected output, and it should not contain any incorrect information.",
    evaluation_steps=[
        "Read the input question.",
        "Read the expected output and the actual output.",
        "Check whether the actual output states the same core answer as the expected output.",
        "If the core answer matches, mark the test as passing; otherwise mark it as failing.",
    ],
    model=evaluator_model,
    threshold=0.5,
    async_mode=True,
    strict_mode=False,
)

completeness_metrics = GEval(
    name="Completeness",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    criteria="Evaluate whether the actual output is complete and accurate compared to the expected output. The actual output should include all the information present in the expected output, and it should not contain any incorrect information.",
    evaluation_steps=[
        "Read the input question.",
        "Read the expected output and the actual output.",
        "Check whether the actual output includes all the information from the expected output.",
        "If the actual output is complete and accurate, mark the test as passing; otherwise mark it as failing.",
    ],
    model=evaluator_model,
    threshold=0.5,
    async_mode=True,
    strict_mode=False,
)

### Final Evaluation with GEval

In [14]:
from deepeval.evaluate import evaluate
from deepeval.evaluate.configs import AsyncConfig
from deepeval.metrics import AnswerRelevancyMetric

evaluate(
    test_cases=dataset.test_cases,
    metrics=[concise_metrics, AnswerRelevancyMetric(), completeness_metrics],
    async_config=AsyncConfig(run_async=False),   # local Ollama runs sequentially
)

✨ You're running DeepEval's latest Conciseness and Correctness [GEval] Metric! (using llama3.2:latest (Ollama), 
strict=False, async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.4, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Completeness [GEval] Metric! (using llama3.2:latest (Ollama), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ What is ShopSphere Assistant?                                                                               │
│  ├──   Input:              What is ShopSphere Assistant?                                                        │
│  │     Actual Output:      ShopSphere Assistant is a conversational AI chatbot embedded across the              │
│  │                         ShopSphere e-commerce platform - web, iOS and Android. It helps shoppers find        │
│  │                         products, track orders, manage returns and other shopping-related tasks.             │
│  │     Expected Output:    ShopSphere Assistant is an AI chatbot on the ShopSphere website and mobile apps      │
│  │                         that helps customers find products, track orders and handle returns using a          │
│  │                         retrieval-augmented generation (RAG) system built on the llama3.2 model with a       │
│  │                         local vector database.                                                               │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                              ┃ Score ┃ Threshold ┃ Reason                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Conciseness and Correctness [GEval] │ 0.60  │ 0.50      │ The actual output does not mention     │
│              │                                     │       │           │ the RAG syst...                        │
│        PASS  │ Answer Relevancy                    │ 1.00  │ 0.50      │ The score is 1.00 because the          │
│              │                                     │       │           │ response stayed f...                   │
│        FAIL  │ Completeness [GEval]                │ 0.40  │ 0.50      │ The actual output is missing the       │
│              │                                     │       │           │ specific model details and RAG         │
│              │                                     │       │           │ system, which are crucial for          │
│              │                                     │       │           │ understanding the technology behind    │
│              │                                     │       │           │ ShopSphere Assistant.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Refund timeline (Passed 3 metrics)                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ Human handoff trigger                                                                                       │
│  ├──   Input:              When is a conversation handed o

⚠ WARNING: No hyperparameters logged.
» ]8;id=6985092;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 62.89s | token cost: 0.016085000000000002 USD)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='What is ShopSphere Assistant?', success=False, metrics_data=[MetricData(name='Conciseness and Correctness [GEval]', threshold=0.5, success=True, score=0.6, reason='The actual output does not mention the RAG system or the llama3.2 model, which are key components of the expected output. The core answer is correct, but the additional details are missing.', strict_mode=False, flaky=False, evaluation_model='llama3.2:latest (Ollama)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Criteria:\nEvaluate whether the actual output is concise and correct compared to the expected output. The actual output should be a brief summary of the expected output, and it should not contain any incorrect information. \n \nEvaluation Steps:\n[\n    "Read the input question.",\n    "Read the expected output and the actual output.",\n    "Check whether the actual output states the same core answer as the expected output.",\n    "If 

### Notes

- The pipeline is identical to the web-based version in `TestingRag.ipynb` - **only the loader changed** (`PyPDFLoader` instead of `WebBaseLoader`).
- The Chroma collection `shopsphere_prd_v23` is named after the document version so older document snapshots stay queryable.
- The prompt now includes a grounding rule: *answer only from context*. This prevents the model from hallucinating about anything outside the PRD.